# AI-асистент продуктового аналітика
**Лабораторна робота №4.** AI-підкріплена дата аналітика · Методи та інструменти продуктової аналітики
**Виконав:** студент групи ІА-33 Супик Андрій

Асистент забирає на себе рутинну частину роботи аналітика: профілювання даних, пошук гіпотез, відповіді на питання природною мовою, розбір змін метрик і текст для стейкхолдера.

**Головний принцип:** усі числа рахує Pandas, а LLM (Google Gemini) лише інтерпретує вже пораховані факти й пише текст або код. Кожна відповідь моделі проходить програмну перевірку: чи існують згадані колонки й сегменти, чи збігаються числа з даними, чи безпечний згенерований код.

| № | Функція | Що робить |
|---|---|---|
| 1 | `auto_eda(df)` | Профіль таблиці (Pandas) + інтерпретація якості даних (LLM) |
| 2 | `suggest_hypotheses(users)` | 5–10 гіпотез, кожна посилається на факти з таблиці фактів |
| 3 | `ask("питання")` | Natural language → код Pandas → виконання → результат (код видно) |
| 4 | `explain_change(metric, ...)` | Розклад зміни метрики по сегментах + гіпотеза причини |
| 5 | `executive_summary(kpis)` | Короткий текст для стейкхолдера з перевіркою чисел |

**Датасет:** дані мобільного застосунку з ЛР3: `registrations` (202 тис. користувачів), `sessions` (1,5 млн сесій), `payments` (11 тис. оплат) і довідники.

## 0. Налаштування
**API-ключ.** Безкоштовний ключ Gemini: [aistudio.google.com](https://aistudio.google.com) → **Get API key → Create API key**.
У Colab найзручніше зберегти його в **Secrets** (іконка 🔑 зліва) з назвою `GEMINI_API_KEY` і увімкнути доступ для notebook. Якщо секрету немає, notebook попросить ввести ключ вручну (він не зберігається у файлі).

In [ ]:
import os, re, json, time, ast, textwrap
from getpass import getpass

import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    try:
        from google.colab import userdata          # Colab Secrets
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception:
        GEMINI_API_KEY = None
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass('Встав Gemini API key: ')
print('Ключ отримано:', bool(GEMINI_API_KEY))

## 1. Клієнт LLM
Модель викликається через REST API Gemini, без додаткових бібліотек.
- **JSON mode** (`responseMimeType: application/json`): модель зобов'язана повернути JSON, тож відповідь розбирається як структура, а не витягується регулярками з тексту.
- **Низька температура (0.2)**: для аналітики потрібна стабільність, а не креативність.
- **Повтори з паузою** при перевантаженні (429/503) і **перехід на іншу модель**, якщо поточна недоступна.
- Усі виклики пишуться в `LLM_LOG`, щоб можна було подивитись промпт, відповідь і час.

In [ ]:
MODELS = ['gemini-2.5-flash', 'gemini-flash-latest', 'gemini-2.0-flash']
API_URL = 'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent'
LLM_LOG = []


class LLMError(RuntimeError):
    pass


def parse_json(text):
    """Розбирає JSON із відповіді моделі (на випадок, якщо вона все ж обгорне його в ```json)."""
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', cleaned, flags=re.S)
        if match:
            return json.loads(match.group(0))
        raise LLMError(f'Модель повернула не JSON: {text[:300]}')


def call_llm(system_prompt, user_prompt, temperature=0.2, max_retries=4):
    """Надсилає промпт у Gemini і повертає розібраний JSON."""
    body = {
        'systemInstruction': {'parts': [{'text': system_prompt}]},
        'contents': [{'role': 'user', 'parts': [{'text': user_prompt}]}],
        'generationConfig': {'temperature': temperature, 'responseMimeType': 'application/json'},
    }
    last_error = None
    for model in MODELS:
        for attempt in range(max_retries):
            started = time.time()
            try:
                resp = requests.post(API_URL.format(model=model), params={'key': GEMINI_API_KEY},
                                     json=body, timeout=180)
            except requests.RequestException as e:
                last_error = e
                time.sleep(3)
                continue
            if resp.status_code == 404:                     # модель недоступна → наступна
                last_error = f'{model}: 404'
                break
            if resp.status_code in (429, 500, 503):          # перевантаження → чекаємо й повторюємо
                last_error = f'{model}: {resp.status_code}'
                time.sleep(5 * (attempt + 1))
                continue
            if resp.status_code != 200:
                raise LLMError(f'{model}: HTTP {resp.status_code}: {resp.text[:300]}')
            data = resp.json()
            candidates = data.get('candidates') or []
            if not candidates or 'content' not in candidates[0]:
                raise LLMError(f'Порожня відповідь моделі: {json.dumps(data)[:300]}')
            text = ''.join(p.get('text', '') for p in candidates[0]['content'].get('parts', []))
            parsed = parse_json(text)
            LLM_LOG.append({'model': model, 'seconds': round(time.time() - started, 1),
                            'system': system_prompt, 'user': user_prompt, 'response': parsed})
            return parsed
    raise LLMError(f'Не вдалося отримати відповідь: {last_error}')


# Перевірка зв'язку
print(call_llm('Return JSON only.', 'Return {"status": "ok"}'))

## 2. Завантаження даних
Файли лежать у папці `data/` репозиторію. Якщо notebook запущено в Colab і папки немає, репозиторій клонується автоматично. Великі таблиці стиснуті в `.csv.gz`, Pandas читає їх напряму.
Будуємо таблицю **`users`**: один рядок на користувача з атрибутами реєстрації та агрегатами по сесіях і оплатах. Саме вона є основою для гіпотез і питань.

In [ ]:
DATA_DIR = 'data'
REPO_URL = 'https://github.com/insxlll/MIPA-4laba'   # репозиторій з папкою data/

if not os.path.exists(DATA_DIR):                      # у Colab даних ще немає → клонуємо репозиторій
    os.system(f'git clone -q --depth 1 {REPO_URL} repo_data')
    DATA_DIR = 'repo_data/data'


def read_table(name, **kwargs):
    for ext in ('.csv', '.csv.gz'):
        path = os.path.join(DATA_DIR, name + ext)
        if os.path.exists(path):
            return pd.read_csv(path, **kwargs)
    raise FileNotFoundError(f'Немає файлу {name}.csv(.gz) у папці {DATA_DIR}/')


registrations = read_table('registrations', parse_dates=['reg_dt'])
payments = read_table('payments', parse_dates=['pay_dt'])
sessions = read_table('sessions', parse_dates=['session_start_dt'])
countries = read_table('country_info_dict').drop_duplicates('country_code')   # у довіднику є дубль коду
features = read_table('feature_type_dict')

payments = payments.merge(features, on='feature_type_id', how='left')

pay_by_user = payments.groupby('user_id').agg(
    revenue=('amount_usd', 'sum'), n_payments=('amount_usd', 'size'), first_pay_dt=('pay_dt', 'min'))
ses_by_user = sessions.groupby('user_id').agg(
    n_sessions=('session_start_dt', 'size'), total_duration_sec=('duration', 'sum'),
    first_session_dt=('session_start_dt', 'min'), last_session_dt=('session_start_dt', 'max'))
ses_by_user['total_duration_min'] = ses_by_user.pop('total_duration_sec') / 60

users = (registrations
         .merge(countries, on='country_code', how='left')
         .merge(pay_by_user, on='user_id', how='left')
         .merge(ses_by_user, on='user_id', how='left'))
users[['revenue', 'n_payments', 'n_sessions', 'total_duration_min']] = \
    users[['revenue', 'n_payments', 'n_sessions', 'total_duration_min']].fillna(0)
users['is_payer'] = users['n_payments'] > 0
users['age_group'] = pd.cut(users['age'], [17, 24, 34, 44, 54, 120],
                            labels=['18-24', '25-34', '35-44', '45-54', '55+'])
users['is_new'] = users['reg_dt'] >= '2024-04-05'      # зареєстровані у періоді даних

TABLES = {'users': users, 'registrations': registrations, 'sessions': sessions,
          'payments': payments, 'countries': countries}
for name, df in TABLES.items():
    print(f'{name:14s} {len(df):>10,} рядків × {df.shape[1]} колонок')

## 3. Перевірки проти галюцинацій
LLM упевнено вигадує числа, яких немає в даних. Тому:
- `check_numbers(text, facts)` витягує всі числа з тексту моделі й шукає їх серед фактично порахованих значень (з урахуванням округлення й відсотків). Числа, яких немає в даних, позначаються ⚠️.
- у функціях нижче окремо перевіряється, що згадані колонки, сегменти та ідентифікатори фактів **реально існують**.

In [ ]:
def _collect_numbers(obj, out):
    if isinstance(obj, dict):
        for v in obj.values():
            _collect_numbers(v, out)
    elif isinstance(obj, (list, tuple, set)):
        for v in obj:
            _collect_numbers(v, out)
    elif isinstance(obj, (int, float, np.integer, np.floating)) and not isinstance(obj, bool):
        if np.isfinite(obj):
            out.append(float(obj))
    elif isinstance(obj, str):
        for n in _extract_numbers(obj):
            out.append(n)
    return out


def _extract_numbers(text):
    text = str(text)
    # дати й час не є «числами» для перевірки
    text = re.sub(r'\d{4}-\d{2}-\d{2}(?:[ T]\d{2}:\d{2}(?::\d{2})?)?', ' ', text)
    text = re.sub(r'\b\d{1,2}\.\d{1,2}\.\d{4}\b', ' ', text)
    pattern = r'(?<![\w.])-?\d{1,3}(?:[ \u00a0]\d{3})+(?![\d])(?:[.,]\d+)?|(?<![\w.])-?\d+(?:[.,]\d+)*'
    nums = []
    for raw in re.findall(pattern, text):
        s = raw.replace('\u00a0', '').replace(' ', '')
        if ',' in s and '.' in s:
            s = s.replace(',', '')
        elif s.count(',') == 1 and len(s.split(',')[1]) != 3:
            s = s.replace(',', '.')
        else:
            s = s.replace(',', '')
        if s.count('.') > 1:
            continue
        try:
            nums.append(float(s))
        except ValueError:
            pass
    return nums


def check_numbers(text, facts, rel_tol=0.02, ignore=(2024, 2025, 2026)):
    """Повертає числа з тексту, яких немає серед фактів (можливі галюцинації)."""
    known = _collect_numbers(facts, [])
    variants = set()
    for k in known:
        variants.update({k, k * 100, k / 1000, k / 1_000_000, round(k), round(k * 100)})
    unverified = []
    for n in _extract_numbers(text):
        if abs(n) <= 10 and float(n).is_integer():      # дрібні цілі: номери пунктів, «топ-5», дні
            continue
        if n in ignore:
            continue
        if not any(abs(n - v) <= max(rel_tol * abs(v), 0.051) for v in variants):
            unverified.append(n)
    return unverified


def show_check(text, facts, label='Перевірка чисел'):
    bad = check_numbers(text, facts)
    if bad:
        display(Markdown(f'⚠️ **{label}:** числа, яких немає в даних: `{bad}`. Їх треба перевірити вручну.'))
    else:
        display(Markdown(f'✅ **{label}:** усі числа в тексті знайдено серед пораховано даних.'))
    return bad

## Функція 1. Автоматичний EDA
**Pandas рахує** профіль кожної колонки: тип, пропуски, кількість унікальних, статистики, викиди за правилом 3×IQR, від'ємні й нульові значення, діапазон дат, пробіли та різний регістр у тексті.
**LLM отримує лише цей профіль** (не сирі дані) і пише висновки про якість даних у форматі JSON.
**Перевірка:** проблеми, що посилаються на неіснуючі колонки, відкидаються; числа в тексті звіряються з профілем.

In [ ]:
EDA_SYSTEM = """You are a senior product data analyst reviewing data quality.
You receive a JSON profile of ONE table computed with pandas. You never see raw rows.
Rules:
- Use ONLY numbers that appear in the profile. Never invent or recompute numbers.
- Every issue must reference an existing column name from the profile.
- Be specific: say which column, what is wrong, and the evidence from the profile.
- Severity: "high" breaks metrics, "medium" may bias results, "low" is cosmetic.
- Write all text in Ukrainian.
Return JSON exactly in this schema:
{"summary": "2-3 sentences about the table and its overall quality",
 "issues": [{"column": "...", "problem": "...", "evidence": "...", "severity": "high|medium|low", "fix": "..."}],
 "observations": ["notable facts about distributions, 3-5 items"],
 "next_steps": ["what an analyst should check next, 2-4 items"]}"""


def profile_table(df, max_top=5):
    prof = {'rows': int(len(df)), 'columns': int(df.shape[1]),
            'duplicate_rows': int(df.duplicated().sum()) if len(df) else 0, 'columns_profile': {}}
    for col in df.columns:
        s = df[col]
        p = {'dtype': str(s.dtype), 'missing': int(s.isna().sum()),
             'missing_pct': round(float(s.isna().mean() * 100), 2) if len(s) else 0.0,
             'unique': int(s.nunique(dropna=True))}
        if pd.api.types.is_bool_dtype(s):
            p['true_pct'] = round(float(s.mean() * 100), 2) if len(s) else None
        elif pd.api.types.is_numeric_dtype(s):
            d = s.dropna()
            if len(d):
                q1, q3 = d.quantile([0.25, 0.75])
                iqr = q3 - q1
                p.update({'min': float(d.min()), 'p25': float(q1), 'median': float(d.median()),
                          'p75': float(q3), 'max': float(d.max()), 'mean': round(float(d.mean()), 3),
                          'negative': int((d < 0).sum()), 'zeros': int((d == 0).sum()),
                          'outliers_3iqr': int(((d < q1 - 3 * iqr) | (d > q3 + 3 * iqr)).sum()) if iqr > 0 else 0})
        elif pd.api.types.is_datetime64_any_dtype(s):
            d = s.dropna()
            if len(d):
                p.update({'min': str(d.min()), 'max': str(d.max()),
                          'top_years': {str(k): int(v) for k, v in d.dt.year.value_counts().head(3).items()}})
        else:
            d = s.dropna().astype(str)
            p['top_values'] = {k: int(v) for k, v in d.value_counts().head(max_top).items()}
            p['values_with_spaces'] = int((d != d.str.strip()).sum())
            p['case_variants'] = int(d.nunique() - d.str.lower().str.strip().nunique())
        prof['columns_profile'][col] = p
    return prof


def auto_eda(df, name='table'):
    """Профіль таблиці (Pandas) + інтерпретація якості даних (LLM)."""
    if df is None or len(df) == 0:
        display(Markdown(f'### EDA: `{name}`\n❌ Таблиця порожня, аналізувати нічого. LLM не викликається.'))
        return None
    prof = profile_table(df)
    display(Markdown(f'### EDA: `{name}` — {prof["rows"]:,} рядків × {prof["columns"]} колонок, '
                     f'дублікатів: {prof["duplicate_rows"]:,}'))
    display(pd.DataFrame(prof['columns_profile']).T.fillna(''))

    report = call_llm(EDA_SYSTEM, f'Table name: {name}\nProfile JSON:\n{json.dumps(prof, ensure_ascii=False, default=str)}')

    issues = [i for i in report.get('issues', []) if i.get('column') in df.columns]
    rejected = [i for i in report.get('issues', []) if i.get('column') not in df.columns]

    md = [f'**Висновок:** {report.get("summary", "")}', '', '**Проблеми якості даних:**']
    icon = {'high': '🔴', 'medium': '🟠', 'low': '🟡'}
    for i in issues:
        md.append(f'- {icon.get(i.get("severity"), "•")} `{i["column"]}`: {i.get("problem", "")} '
                  f'*Доказ:* {i.get("evidence", "")} *Що зробити:* {i.get("fix", "")}')
    md += ['', '**Спостереження:**'] + [f'- {o}' for o in report.get('observations', [])]
    md += ['', '**Що перевірити далі:**'] + [f'- {n}' for n in report.get('next_steps', [])]
    display(Markdown('\n'.join(md)))
    if rejected:
        display(Markdown(f'⚠️ Відкинуто {len(rejected)} проблем(и) з неіснуючими колонками: '
                         f'{[i.get("column") for i in rejected]}'))
    show_check(json.dumps(report, ensure_ascii=False), prof)
    return {'profile': prof, 'report': report}

In [ ]:
eda_users = auto_eda(registrations, 'registrations')

In [ ]:
eda_sessions = auto_eda(sessions, 'sessions')

## Функція 2. Генерація гіпотез
**Pandas будує таблицю фактів:** конверсію в оплату, ARPU, сесії й тривалість у розрізах платформи, статі, віку, країни, тижня реєстрації та дня тижня. Кожен факт має свій `fact_id`.
**LLM пропонує гіпотези й посилається тільки на `fact_id`**, а не пише числа сам. Числа в доказах підставляються з таблиці фактів програмно, тому вигадати їх модель не може. Гіпотези з неіснуючими `fact_id` відкидаються.

In [ ]:
HYPOTHESES_SYSTEM = """You are a senior product analyst. You receive a table of facts computed with pandas
about a freemium mobile app (users, sessions, payments). Each fact has a fact_id.
Task: propose {n} hypotheses worth investigating.
Rules:
- Each hypothesis must be supported by 1-3 fact_ids from the table. Cite ONLY existing fact_ids.
- Do not write numbers yourself: the system inserts the values for the cited fact_ids.
- Compare a segment with its baseline (fact rows with segment "ALL") or with other segments.
- Prefer big, actionable differences; ignore segments with fewer than 500 users.
- Mention possible data-quality explanations when a pattern looks like a tracking artifact.
- Write all text in Ukrainian.
Return JSON: {"hypotheses": [{"title": "short", "hypothesis": "1-2 sentences",
 "fact_ids": ["F001"], "why_it_matters": "...", "how_to_check": "concrete analysis to run",
 "priority": "high|medium|low"}]}"""


def build_fact_table(u, s, min_users=500):
    rows = []

    def add(dimension, segment, metric, value, n_users):
        rows.append({'dimension': dimension, 'segment': str(segment), 'metric': metric,
                     'value': round(float(value), 4), 'n_users': int(n_users)})

    def seg_metrics(df, dimension, segment):
        n = len(df)
        add(dimension, segment, 'users', n, n)
        add(dimension, segment, 'payer_rate_pct', df['is_payer'].mean() * 100, n)
        add(dimension, segment, 'arpu_usd', df['revenue'].mean(), n)
        add(dimension, segment, 'sessions_per_user', df['n_sessions'].mean(), n)
        add(dimension, segment, 'minutes_per_user', df['total_duration_min'].mean(), n)

    seg_metrics(u, 'overall', 'ALL')
    for dim in ['app', 'gender', 'age_group', 'country_name', 'is_new']:
        for seg, part in u.groupby(dim, observed=True):
            if len(part) >= min_users:
                seg_metrics(part, dim, seg)

    new = u[u['is_new']]
    for week, part in new.groupby(new['reg_dt'].dt.to_period('W').dt.start_time):
        add('reg_week', week.date(), 'users', len(part), len(part))
        add('reg_week', week.date(), 'payer_rate_pct', part['is_payer'].mean() * 100, len(part))

    daily = s.groupby(s['session_start_dt'].dt.date)['user_id'].nunique()
    by_dow = daily.groupby(pd.to_datetime(daily.index).day_name()).mean()
    for dow, v in by_dow.items():
        add('weekday', dow, 'avg_dau', v, v)

    facts = pd.DataFrame(rows)
    facts.insert(0, 'fact_id', [f'F{i:03d}' for i in range(1, len(facts) + 1)])
    return facts


def suggest_hypotheses(u=None, s=None, n=7):
    u = users if u is None else u
    s = sessions if s is None else s
    if len(u) == 0:
        display(Markdown('❌ Немає даних для гіпотез.'))
        return None
    facts = build_fact_table(u, s)
    display(Markdown(f'**Таблиця фактів:** {len(facts)} фактів (показано перші 15)'))
    display(facts.head(15))

    result = call_llm(HYPOTHESES_SYSTEM.replace('{n}', str(n)), 'Facts:\n' + facts.to_csv(index=False))
    fact_index = facts.set_index('fact_id')

    md, kept, dropped = [], [], []
    for h in result.get('hypotheses', []):
        ids = [f for f in h.get('fact_ids', []) if f in fact_index.index]
        missing = [f for f in h.get('fact_ids', []) if f not in fact_index.index]
        if not ids:
            dropped.append(h.get('title'))
            continue
        kept.append(h)
        evidence = '; '.join(
            f'{fact_index.loc[f, "dimension"]}={fact_index.loc[f, "segment"]}: '
            f'{fact_index.loc[f, "metric"]} = {fact_index.loc[f, "value"]:,.2f}' for f in ids)
        md.append(f'### {len(kept)}. {h.get("title", "")}  `{h.get("priority", "")}`\n'
                  f'{h.get("hypothesis", "")}\n\n'
                  f'- **Факти з даних (підставлено програмно):** {evidence}\n'
                  f'- **Чому важливо:** {h.get("why_it_matters", "")}\n'
                  f'- **Як перевірити:** {h.get("how_to_check", "")}'
                  + (f'\n- ⚠️ Відкинуто неіснуючі fact_id: {missing}' if missing else ''))
    display(Markdown('\n\n'.join(md)))
    if dropped:
        display(Markdown(f'⚠️ Відкинуто гіпотези без жодного реального факту: {dropped}'))
    show_check(json.dumps(kept, ensure_ascii=False), facts['value'].tolist(), 'Перевірка чисел у тексті гіпотез')
    return {'facts': facts, 'hypotheses': kept}

In [ ]:
hyp = suggest_hypotheses(n=7)

## Функція 3. Natural Language → Pandas
Користувач питає звичайною мовою. LLM отримує **схему таблиць** (колонки, типи, 3 приклади рядків) і повертає код Pandas.
- **Код завжди показується**, щоб аналітик бачив логіку.
- **Перевірка безпеки (AST):** заборонені `import`, доступ до файлів, мережі, `eval/exec`, «приховані» атрибути `__...__`. Небезпечний код не виконується.
- Код виконується на **копіях** таблиць, тому оригінальні дані не зміняться.
- Якщо код падає з помилкою, модель отримує текст помилки й **один раз виправляє** код сама.
- Якщо питання не про дані, модель повертає `can_answer: false` і пояснення.

In [ ]:
NL2PANDAS_SYSTEM = """You translate analytics questions into pandas code.
Available pandas DataFrames (already loaded) and their schema are given below.
Rules:
- Use only pandas (pd), numpy (np) and the listed DataFrames. No imports, no file or network access, no plotting.
- Do not modify the DataFrames in place; build new objects.
- Store the final answer in a variable named result (a DataFrame, Series or scalar). Keep it small: at most 50 rows.
- Use the exact column names from the schema. Dates are pandas datetime64.
- If the question cannot be answered with these tables (or is not about the data), set can_answer to false and explain why.
- Write explanation and assumptions in Ukrainian.
Return JSON: {"can_answer": true, "code": "...", "explanation": "what the code does, 1-2 sentences", "assumptions": ["..."]}

Schema:
{schema}"""

INTERPRET_SYSTEM = """You are a product analyst. Given a question and the computed result (from pandas),
write a short answer in Ukrainian (2-3 sentences). Use only numbers present in the result. Return JSON: {"answer": "..."}"""

FORBIDDEN_NAMES = {'open', 'exec', 'eval', 'compile', '__import__', 'globals', 'locals', 'vars',
                   'getattr', 'setattr', 'delattr', 'input', 'os', 'sys', 'subprocess', 'shutil',
                   'requests', 'pathlib', 'builtins', 'breakpoint'}
FORBIDDEN_ATTRS = {'to_csv', 'to_excel', 'to_pickle', 'to_parquet', 'to_sql', 'read_csv', 'read_sql',
                   'read_pickle', 'system', 'remove', 'unlink', 'rmtree'}


def table_schema(tables):
    parts = []
    for name, df in tables.items():
        cols = ', '.join(f'{c} ({t})' for c, t in df.dtypes.astype(str).items())
        sample = df.head(3).to_csv(index=False)
        parts.append(f'DataFrame `{name}`: {len(df):,} rows\nColumns: {cols}\nSample:\n{sample}')
    return '\n'.join(parts)


def check_code_safety(code):
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f'Синтаксична помилка: {e}'
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            return 'Імпорти заборонені'
        if isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            return f'Заборонене ім\'я: {node.id}'
        if isinstance(node, ast.Attribute):
            if node.attr.startswith('__') or node.attr in FORBIDDEN_ATTRS:
                return f'Заборонений атрибут: {node.attr}'
    return None


def run_code(code, tables):
    namespace = {'pd': pd, 'np': np, **{k: v.copy() for k, v in tables.items()}}
    exec(code, {'__builtins__': {'len': len, 'range': range, 'min': min, 'max': max, 'sum': sum,
                                 'round': round, 'abs': abs, 'sorted': sorted, 'list': list,
                                 'dict': dict, 'str': str, 'int': int, 'float': float,
                                 'zip': zip, 'enumerate': enumerate, 'set': set, 'tuple': tuple,
                                 'bool': bool, 'any': any, 'all': all,
                                 'isinstance': isinstance, 'print': print, 'map': map, 'reversed': reversed}}, namespace)
    if 'result' not in namespace:
        raise ValueError('Код не створив змінну result')
    return namespace['result']


def ask(question, tables=None, interpret=True):
    tables = TABLES if tables is None else tables
    display(Markdown(f'### ❓ {question}'))
    if not question or not question.strip():
        display(Markdown('❌ Порожнє питання.'))
        return None
    system = NL2PANDAS_SYSTEM.replace('{schema}', table_schema(tables))
    plan = call_llm(system, f'Question: {question}')

    if not plan.get('can_answer', False) or not plan.get('code'):
        display(Markdown(f'🚫 **Асистент не може відповісти на це питання за даними:** {plan.get("explanation", "")}'))
        return plan

    code = plan['code']
    for attempt in range(2):
        display(Markdown(f'**Згенерований код{" (виправлений)" if attempt else ""}:**\n```python\n{code}\n```'))
        problem = check_code_safety(code)
        if problem:
            display(Markdown(f'⛔ **Код не виконано з міркувань безпеки:** {problem}'))
            return {**plan, 'blocked': problem}
        try:
            result = run_code(code, tables)
            break
        except Exception as e:
            error = f'{type(e).__name__}: {e}'
            display(Markdown(f'⚠️ Помилка виконання: `{error}`'))
            if attempt == 1:
                return {**plan, 'error': error}
            fix = call_llm(system, f'Question: {question}\nYour code:\n{code}\nIt failed with: {error}\n'
                                   f'Return corrected JSON in the same schema.')
            code = fix.get('code') or code

    display(Markdown(f'**Що робить код:** {plan.get("explanation", "")}'))
    if plan.get('assumptions'):
        display(Markdown('**Припущення:** ' + '; '.join(plan['assumptions'])))
    display(result if isinstance(result, (pd.DataFrame, pd.Series)) else Markdown(f'**Результат:** `{result}`'))

    answer = None
    if interpret:
        preview = result.head(20).to_string() if isinstance(result, (pd.DataFrame, pd.Series)) else str(result)
        answer = call_llm(INTERPRET_SYSTEM, f'Question: {question}\nResult:\n{preview}').get('answer')
        display(Markdown(f'**Відповідь:** {answer}'))
        show_check(answer, preview)
    return {'code': code, 'result': result, 'answer': answer}

In [ ]:
r1 = ask('Покажи топ-10 країн за виручкою і кількістю платників')

In [ ]:
r2 = ask('Яка конверсія в оплату (частка платників) на кожній платформі серед нових користувачів?')

In [ ]:
r3 = ask('Скільки в середньому сесій і хвилин на користувача у платників і неплатників?')

## Функція 4. Пояснення зміни метрики
На вхід: метрика і два періоди. **Pandas рахує** значення метрики в кожному сегменті (платформа, країна, стать, вік) в обох періодах і **внесок кожного сегмента** у загальну зміну.
Якщо періоди різної довжини, функція попереджає і **порівнює середнє за день**, бо інакше неповний тиждень виглядатиме як падіння.
**LLM формулює гіпотезу причини** лише на основі найбільших внесків; згадані сегменти звіряються з даними.

In [ ]:
CHANGE_SYSTEM = """You are a product analyst explaining why a metric changed between two periods.
You receive a pandas decomposition: total change and the segments with the largest contribution.
Rules:
- Use only numbers from the input. Do not invent segments that are not in the input.
- Distinguish real business change from artifacts (unequal period length, tracking changes, data cut-off).
- Write in Ukrainian.
Return JSON: {"headline": "one sentence", "drivers": [{"dimension": "...", "segment": "...", "explanation": "..."}],
 "hypotheses": ["possible causes, 2-4"], "next_checks": ["what to verify, 2-3"]}"""

METRICS = {
    'revenue':      ('payments', 'pay_dt', lambda d: d['amount_usd'].sum()),
    'payers':       ('payments', 'pay_dt', lambda d: d['user_id'].nunique()),
    'new_users':    ('registrations', 'reg_dt', lambda d: d['user_id'].nunique()),
    'active_users': ('sessions', 'session_start_dt', lambda d: d['user_id'].nunique()),
}


def _period(df, col, start, end):
    return df[(df[col] >= pd.Timestamp(start)) & (df[col] < pd.Timestamp(end) + pd.Timedelta(days=1))]


def explain_change(metric, previous, current, dims=('app', 'country_name', 'gender', 'age_group'), top=8):
    if metric not in METRICS:
        display(Markdown(f'❌ Невідома метрика `{metric}`. Доступні: {list(METRICS)}'))
        return None
    table, date_col, fn = METRICS[metric]
    base = TABLES[table]
    need = [c for c in ['app', 'country_name', 'gender', 'age_group'] if c not in base.columns]
    base = base.merge(users[['user_id'] + need], on='user_id', how='left') if need else base
    prev, cur = _period(base, date_col, *previous), _period(base, date_col, *current)
    if len(prev) == 0 or len(cur) == 0:
        display(Markdown('❌ В одному з періодів немає даних. Перевір дати.'))
        return None

    days_prev = (pd.Timestamp(previous[1]) - pd.Timestamp(previous[0])).days + 1
    days_cur = (pd.Timestamp(current[1]) - pd.Timestamp(current[0])).days + 1
    v_prev, v_cur = fn(prev), fn(cur)
    summary = {'metric': metric, 'previous_period': list(previous), 'current_period': list(current),
               'days_previous': days_prev, 'days_current': days_cur,
               'value_previous': round(float(v_prev), 2), 'value_current': round(float(v_cur), 2),
               'change_pct': round(float((v_cur / v_prev - 1) * 100), 2) if v_prev else None,
               'per_day_previous': round(float(v_prev / days_prev), 2),
               'per_day_current': round(float(v_cur / days_cur), 2)}
    summary['per_day_change_pct'] = round((summary['per_day_current'] / summary['per_day_previous'] - 1) * 100, 2)

    display(Markdown(f'### Зміна `{metric}`: {previous[0]}…{previous[1]} → {current[0]}…{current[1]}'))
    if days_prev != days_cur:
        display(Markdown(f'⚠️ Періоди різної довжини ({days_prev} і {days_cur} дн.). Тому порівнюємо **середнє за день**: '
                         f'{summary["per_day_previous"]:,.2f} → {summary["per_day_current"]:,.2f} '
                         f'({summary["per_day_change_pct"]:+.1f}%), а не суму ({summary["change_pct"]:+.1f}%).'))

    rows = []
    for dim in dims:
        for seg in set(prev[dim].dropna().unique()) | set(cur[dim].dropna().unique()):
            a = fn(prev[prev[dim] == seg]) / days_prev
            b = fn(cur[cur[dim] == seg]) / days_cur
            rows.append({'dimension': dim, 'segment': str(seg), 'per_day_previous': round(float(a), 2),
                         'per_day_current': round(float(b), 2), 'delta_per_day': round(float(b - a), 2)})
    seg = pd.DataFrame(rows)
    total_delta = summary['per_day_current'] - summary['per_day_previous']
    seg['share_of_change_pct'] = (seg['delta_per_day'] / total_delta * 100).round(1) if total_delta else 0.0
    top_seg = seg.reindex(seg['delta_per_day'].abs().sort_values(ascending=False).index).head(top)
    display(pd.DataFrame([summary]).T.rename(columns={0: 'значення'}))
    display(top_seg)

    report = call_llm(CHANGE_SYSTEM, json.dumps({'summary': summary, 'top_segments': top_seg.to_dict('records')},
                                                ensure_ascii=False, default=str))
    valid = set(zip(seg['dimension'], seg['segment']))
    md = [f'**{report.get("headline", "")}**', '', '**Основні драйвери:**']
    for d in report.get('drivers', []):
        mark = '' if (d.get('dimension'), str(d.get('segment'))) in valid else ' ⚠️ *(сегмента немає в даних)*'
        md.append(f'- `{d.get("dimension")}={d.get("segment")}`{mark}: {d.get("explanation", "")}')
    md += ['', '**Можливі причини:**'] + [f'- {h}' for h in report.get('hypotheses', [])]
    md += ['', '**Що перевірити:**'] + [f'- {c}' for c in report.get('next_checks', [])]
    display(Markdown('\n'.join(md)))
    show_check(json.dumps(report, ensure_ascii=False), {'s': summary, 'seg': seg.to_dict('records')})
    return {'summary': summary, 'segments': seg, 'report': report}

In [ ]:
# Чому «впала» виручка останнього тижня? (тиждень неповний: дані закінчуються 11.05)
chg = explain_change('revenue', previous=('2024-04-29', '2024-05-05'), current=('2024-05-06', '2024-05-11'))

## Функція 5. Executive summary
**Pandas рахує набір KPI** за період і зміну до попереднього періоду. **LLM пише короткий бізнес-текст:** що сталося, чому це важливо, що робити далі.
Після цього **кожне число в тексті звіряється з KPI**. Якщо модель написала число, якого немає в даних, воно позначається ⚠️.

In [ ]:
SUMMARY_SYSTEM = """You write an executive summary for a product manager.
You receive KPIs computed with pandas for the current and previous period.
Rules:
- Use ONLY numbers from the KPI JSON (you may round them). Never compute new numbers.
- Business language, no jargon. Max 120 words in total.
- Structure: what happened -> why it matters -> what to do next.
- Write in Ukrainian.
Return JSON: {"headline": "one sentence", "what_happened": ["2-4 bullets"], "why_it_matters": "1-2 sentences", "next_steps": ["2-3 bullets"]}"""


def compute_kpis(start, end, prev_start, prev_end):
    def block(a, b):
        reg = _period(registrations, 'reg_dt', a, b)
        pay = _period(payments, 'pay_dt', a, b)
        ses = _period(sessions, 'session_start_dt', a, b)
        days = (pd.Timestamp(b) - pd.Timestamp(a)).days + 1
        k = {'days': days, 'new_users': int(reg['user_id'].nunique()),
             'active_users': int(ses['user_id'].nunique()),
             'avg_dau': round(float(ses.groupby(ses['session_start_dt'].dt.date)['user_id'].nunique().mean()), 1) if len(ses) else 0,
             'revenue_usd': round(float(pay['amount_usd'].sum()), 2),
             'payers': int(pay['user_id'].nunique()),
             'avg_check_usd': round(float(pay['amount_usd'].mean()), 2) if len(pay) else 0}
        k['revenue_per_day_usd'] = round(k['revenue_usd'] / days, 2)
        return k
    cur, prev = block(start, end), block(prev_start, prev_end)
    change = {k: round((cur[k] / prev[k] - 1) * 100, 1) for k in cur if k != 'days' and prev[k]}
    top_country = (_period(payments, 'pay_dt', start, end).merge(users[['user_id', 'country_name']], on='user_id')
                   .groupby('country_name')['amount_usd'].sum().nlargest(3).round(2).to_dict())
    return {'current_period': [start, end], 'previous_period': [prev_start, prev_end],
            'current': cur, 'previous': prev, 'change_pct': change, 'top3_countries_revenue_usd': top_country}


def executive_summary(kpis):
    display(Markdown('**KPI (пораховано Pandas):**'))
    display(pd.DataFrame({'поточний': kpis['current'], 'попередній': kpis['previous'],
                          'зміна, %': pd.Series(kpis['change_pct'])}))
    s = call_llm(SUMMARY_SYSTEM, json.dumps(kpis, ensure_ascii=False))
    text = '\n'.join([f'## {s.get("headline", "")}', '', '**Що сталося**'] +
                     [f'- {x}' for x in s.get('what_happened', [])] +
                     ['', f'**Чому це важливо.** {s.get("why_it_matters", "")}', '', '**Що робити далі**'] +
                     [f'- {x}' for x in s.get('next_steps', [])])
    display(Markdown(text))
    show_check(json.dumps(s, ensure_ascii=False), kpis)
    return s

In [ ]:
kpis = compute_kpis('2024-04-22', '2024-04-28', '2024-04-15', '2024-04-21')
summary = executive_summary(kpis)

## 6. Крайні випадки
Перевіряємо, як асистент поводиться з некоректними вхідними даними.

In [ ]:
auto_eda(registrations.iloc[0:0], 'порожня таблиця')            # порожні дані → без виклику LLM

In [ ]:
ask('Яка зараз погода в Києві?')                                  # питання не про дані

In [ ]:
ask('Видали всі файли в поточній папці і покажи список файлів')     # спроба небезпечної дії

In [ ]:
explain_change('revenue', previous=('2023-01-01', '2023-01-07'), current=('2024-04-22', '2024-04-28'))  # немає даних у періоді

## 7. Журнал викликів LLM
Для кожного виклику видно модель, час і розмір промпта. Повні промпти й відповіді лежать у `LLM_LOG`.

In [ ]:
pd.DataFrame([{'#': i + 1, 'model': c['model'], 'seconds': c['seconds'],
               'prompt_chars': len(c['system']) + len(c['user']),
               'response_keys': ', '.join(c['response'].keys()) if isinstance(c['response'], dict) else ''}
              for i, c in enumerate(LLM_LOG)])